## Setup — run this first

Mounts Drive, points the notebook at your project folder, and installs
what's missing. No git, no tokens.

**Your Drive folder must look like this:**

```
MyDrive/Ghana_Dropout_Project_R02/
├── config.py          <- these three at the TOP level,
├── losses.py             not inside notebooks/
├── pipeline.py
├── requirements.txt
├── notebooks/         <- the 11 notebooks
└── data-raw/
    └── ghana_dropout_study_M.xlsx
```

`results/`, `figures/`, `models/` and `data-processed/` are created for you.

Drive saves as it goes, so there is nothing to push — but see the checklist
in the last cell before you submit.


In [1]:
# ============================================================
# SETUP — Google Drive. Run first. Safe to re-run.
# ============================================================
import os, sys, subprocess
from pathlib import Path

PROJECT = "/content/drive/MyDrive/Ghana_Dropout_Project_R02"   # <-- edit if yours differs
RAW_XLSX_NAME = "ghana_dropout_study_M.xlsx"

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")

    root = Path(PROJECT)
    if not root.exists():
        raise FileNotFoundError(
            f"{PROJECT} does not exist.\n"
            "Create that folder in My Drive and put config.py, losses.py, "
            "pipeline.py, requirements.txt, the notebooks/ folder and "
            "data-raw/ inside it."
        )

    # the three modules must sit at the project root, not in notebooks/
    missing = [m for m in ("config.py", "losses.py", "pipeline.py")
               if not (root / m).exists()]
    if missing:
        stray = [m for m in missing if (root / "notebooks" / m).exists()]
        msg = f"Missing from {PROJECT}: {missing}"
        if stray:
            msg += (f"\n{stray} are in notebooks/ instead. Move them UP one "
                    "level, into the project folder itself. If they stay in "
                    "notebooks/, that folder gets treated as the project root "
                    "and results/ is written in the wrong place.")
        raise FileNotFoundError(msg)

    os.chdir(root)
    os.environ["DROPOUT_REPO"] = str(root)

    # Forget any previously loaded copy of the project modules. Python keeps
    # the first version it imported for the whole session, so an edited
    # config.py is silently ignored until the runtime restarts. This makes
    # every run use the files currently in Drive.
    for _m in ("config", "losses", "pipeline"):
        sys.modules.pop(_m, None)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

    # ---- dependencies: only install what is actually missing ------------
    need = []
    for mod, pkg in [("lightgbm", "lightgbm"), ("shap", "shap"),
                     ("catboost", "catboost"), ("xgboost", "xgboost"),
                     ("imblearn", "imbalanced-learn"), ("openpyxl", "openpyxl")]:
        try:
            __import__(mod)
        except ImportError:
            need.append(pkg)
    if need:
        print("installing:", need)
        subprocess.run(f"pip install -q {' '.join(need)}", shell=True)
    else:
        print("all dependencies present")

    # ---- raw workbook ---------------------------------------------------
    (root / "data-raw").mkdir(exist_ok=True)
    xlsx = root / "data-raw" / RAW_XLSX_NAME
    if xlsx.exists():
        print(f"raw workbook: {xlsx.name}")
    else:
        loose = list(root.glob(RAW_XLSX_NAME)) + list(root.glob(f"**/{RAW_XLSX_NAME}"))
        if loose:
            import shutil
            shutil.copy(loose[0], xlsx)
            print(f"copied {loose[0]} -> data-raw/")
        else:
            print(f"NOT FOUND: data-raw/{RAW_XLSX_NAME}\n"
                  "Notebook 1 needs it. Notebooks 2-9 read "
                  "data-processed/cleaned_data.csv instead and are fine "
                  "without it.")

    print(f"\nPROJECT : {os.getcwd()}")
else:
    print("Not in Colab — paths resolve from the project root.")


Mounted at /content/drive
installing: ['catboost']
raw workbook: ghana_dropout_study_M.xlsx

PROJECT : /content/drive/MyDrive/Ghana_Dropout_Project_R02


# Notebook 9 — Negative-Result Diagnostic

Answers GATE-4 and Q23 through Q26: is the null a finding, an artefact, or
unestablished?

## What changed from R01

| Change | Reason |
|---|---|
| Arms differ **only in the objective** | R01's ten-seed run compared `RAW_FEATURE_COLS` against all 41 columns, so Table 5 carried the same confound as Table 3 |
| Same pipeline as the headline | Q6: R01's ten-seed protocol was a different pipeline reported in a different table, so it was not commensurable |
| **All three diagnostic CSVs committed** | Q4: Table 5, the MDE of 0.0131 and the zero-flip claim traced to nothing in the repository |
| Leave-one-**school**-out added beside leave-one-seed-out | Q25 C2: the condition asks for a fold or dataset removed; R01 removed a seed, which is a different thing |
| Q23's eight conditions scored from data | GATE-4 |
| Q24 routing computed, not asserted | Q24 |
| The notebook is called 9, not 9b | Q4: M6, M14 and R3 cite "Notebook 9b" three times; no such file exists |

## The wording that matters

The result is **unestablished**, not disproven. That distinction is exact and
it is not a criticism: it means the testbed has not yet been shown capable of
detecting the effect it reports as absent. Say "unestablished" in the viva,
never "we found no effect" — the first keeps you working, the second invites
an examiner to test a claim you cannot defend.

In [2]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from config import *
from pipeline import (preprocess_inside_fold, frozen_split, cv_splits,
                      run_grid, compare_arms, two_level_variance,
                      school_splits, fit_arm, score_binary,
                      ARMS, ARM_LABELS, HEADLINE, OLD_HEADLINE)

banner("NOTEBOOK 9 — NEGATIVE-RESULT DIAGNOSTIC")
OUT = run_dir("notebook09_diagnostic")
capture_environment(OUT)

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)
N_POS = int(train_pool[TARGET].sum())
BASE = float(train_pool[TARGET].mean())
print(f"train_pool {train_pool.shape}, {N_POS} dropout ({100*BASE:.1f}%)")
print(f"\narms: {HEADLINE[0]} vs {HEADLINE[1]} — identical feature matrix, "
      "identical weighting, one argument different")

NOTEBOOK 9 — NEGATIVE-RESULT DIAGNOSTIC
repo            : /content/drive/MyDrive/Ghana_Dropout_Project_R02
provenance      : NONE — set FREEZE_TAG in config.py before scoring the test set
school_handling : drop
FEATURE SET     : records_plus_questionnaire   (SECONDARY — includes friend-reported questionnaire items)
primary metric  : auc_pr
train_pool (784, 42), 69 dropout (8.8%)

arms: E_all_ce_noW vs G_all_focal_noW — identical feature matrix, identical weighting, one argument different


In [4]:
# ---- 1. ten seeds, matched arms, same pipeline as the headline -------
fold_df, preds = run_grid(train_pool, seeds=SEEDS,
                          arms=list(HEADLINE) + [OLD_HEADLINE[0]],
                          collect_predictions=True)
fold_df.to_csv(OUT / "diagnostic_fold_scores.csv", index=False)

per_seed, summ = compare_arms(fold_df, HEADLINE[0], HEADLINE[1],
                              "matched: focal vs cross-entropy")
per_seed.to_csv(OUT / "diagnostic_10_seed_stability.csv", index=False)   # COMMITTED
print("TEN-SEED STABILITY (matched arms)\n")
print(per_seed[["seed", "ref_mean", "test_mean", "diff_mean", "sign",
                "wilcoxon_p", "cohens_d", "ci95_lo", "ci95_hi"]]
      .round(4).to_string(index=False))
print(f"\ngrand mean {summ['grand_mean_diff']:+.5f}  "
      f"between-seed SD {summ['between_seed_sd']:.5f}")
print(f"favouring focal {summ['n_seeds_favouring_test']}/{summ['n_seeds']}, "
      f"favouring reference {summ['n_seeds_favouring_ref']}/{summ['n_seeds']}")
print(f"seeds with p < {ALPHA_LEVEL}: {summ['n_seeds_p_below_alpha']}")

# also report the confounded contrast, for comparison with R01's Table 5
_, summ_old = compare_arms(fold_df, OLD_HEADLINE[0], OLD_HEADLINE[1],
                           "as published (3 changes)")
print(f"\nfor comparison, the R01 contrast (3 changes): "
      f"{summ_old['grand_mean_diff']:+.5f}")
print(f"difference between the two framings: "
      f"{summ['grand_mean_diff']-summ_old['grand_mean_diff']:+.5f} "
      "— this is how much of R01's headline was NOT the loss function")

  seed 42 (1/10) [36s]
  seed 123 (2/10) [62s]
  seed 456 (3/10) [86s]
  seed 789 (4/10) [107s]
  seed 1024 (5/10) [131s]
  seed 2048 (6/10) [152s]
  seed 3333 (7/10) [175s]
  seed 5555 (8/10) [197s]
  seed 7777 (9/10) [219s]
  seed 9999 (10/10) [240s]
TEN-SEED STABILITY (matched arms)

 seed  ref_mean  test_mean  diff_mean sign  wilcoxon_p  cohens_d  ci95_lo  ci95_hi
   42    0.9925     0.9891    -0.0034    -      0.0164   -0.2031  -0.0071  -0.0007
  123    0.9871     0.9833    -0.0039    -      0.0293   -0.1420  -0.0082  -0.0006
  456    0.9854     0.9841    -0.0013    -      0.1484   -0.0748  -0.0029   0.0001
  789    0.9796     0.9776    -0.0020    -      0.2890   -0.0564  -0.0050   0.0008
 1024    0.9875     0.9836    -0.0039    -      0.0342   -0.1559  -0.0068  -0.0013
 2048    0.9849     0.9835    -0.0014    -      0.2470   -0.0603  -0.0042   0.0011
 3333    0.9892     0.9884    -0.0008    -      0.7749   -0.0466  -0.0032   0.0016
 5555    0.9898     0.9869    -0.0029    -      

In [5]:
# ---- 2. power analysis — PAIRED, because the design is paired -----------
# Both arms run on the SAME folds, so what matters is how much the
# DIFFERENCE varies from fold to fold, not how much each arm varies on its
# own. Fold scores for the two arms rise and fall together, so the paired
# differences vary far less than either arm. Using the unpaired SDs (as R01
# did, and as the first version of this notebook did) overstates the MDE
# several-fold and wrongly declares a stable effect "underpowered".
from scipy.stats import norm
wide = fold_df.pivot_table(index=["seed", "fold"], columns="arm", values="auc_pr")
paired = (wide[HEADLINE[1]] - wide[HEADLINE[0]]).dropna()
n_folds = N_SPLITS * N_REPEATS
sd_paired = float(paired.groupby(level="seed").std(ddof=1).mean())
z_a, z_b = norm.ppf(1 - ALPHA_LEVEL/2), norm.ppf(0.80)
MDE = (z_a + z_b) * sd_paired / np.sqrt(n_folds)      # per seed: conservative
observed = abs(summ["grand_mean_diff"])

# the old, wrong calculation, printed for the M6/M16 correction
sd_ref = fold_df[fold_df["arm"] == HEADLINE[0]].groupby("seed")["auc_pr"].std().mean()
sd_test = fold_df[fold_df["arm"] == HEADLINE[1]].groupby("seed")["auc_pr"].std().mean()
MDE_unpaired = (z_a + z_b) * float(np.sqrt((sd_ref**2 + sd_test**2) / 2)) / np.sqrt(n_folds)

power_df = pd.DataFrame([{
    "design": "paired (same folds, both arms)",
    "n_paired_folds_per_seed": n_folds, "n_seeds": len(SEEDS),
    "alpha": ALPHA_LEVEL, "target_power": 0.80,
    "sd_of_paired_differences": sd_paired, "mde_paired": MDE,
    "mde_unpaired_WRONG": MDE_unpaired,
    "observed_abs_diff": observed,
    "underpowered": bool(observed < MDE),
}])
power_df.to_csv(OUT / "diagnostic_power_analysis.csv", index=False)   # COMMITTED
print(f"SD of paired differences : {sd_paired:.4f}")
print(f"MDE (paired, per seed)   : {MDE:.4f}")
print(f"observed effect          : {observed:.4f}")
print(f"-> {'UNDERPOWERED' if observed < MDE else 'ADEQUATELY POWERED'} "
      f"at the single-seed level")
ratio_txt = f"{MDE_unpaired/MDE:.1f}x too large. " if MDE > 0 else ""
print(f"\nfor comparison, the unpaired calculation gives MDE {MDE_unpaired:.4f} — "
      + ratio_txt + "That is the calculation R01 used; correct it in the manuscript.")
print("\nThe 10-seed replication is further evidence beyond this single-seed "
      "power figure. Seeds re-partition the SAME 784 pupils, so they are not "
      "independent samples and must not be pooled into one big n.")

SD of paired differences : 0.0070
MDE (paired, per seed)   : 0.0039
observed effect          : 0.0022
-> UNDERPOWERED at the single-seed level

for comparison, the unpaired calculation gives MDE 0.0127 — 3.2x too large. That is the calculation R01 used; correct it in the manuscript.

The 10-seed replication is further evidence beyond this single-seed power figure. Seeds re-partition the SAME 784 pupils, so they are not independent samples and must not be pooled into one big n.


In [6]:
# ---- 3. leave-one-seed-out AND leave-one-school-out (C2) ------------
full = per_seed["diff_mean"].mean()
full_sign = "+" if full > 0 else "-"
loo = []
for i, r in per_seed.iterrows():
    rem = per_seed.drop(i)["diff_mean"].mean()
    loo.append({"removed_seed": int(r["seed"]), "remaining_mean_diff": rem,
                "sign": "+" if rem > 0 else "-",
                "flipped": ("+" if rem > 0 else "-") != full_sign})
loo = pd.DataFrame(loo)
loo.to_csv(OUT / "diagnostic_leave_one_seed_out.csv", index=False)   # COMMITTED
print(f"full mean {full:+.5f} ({full_sign})")
print(loo.round(5).to_string(index=False))
print(f"sign flips: {int(loo['flipped'].sum())}/{len(loo)}")

# Q25 C2 asks for a FOLD or DATASET removed. Removing a seed is neither.
splits = school_splits(df)
logo = []
if splits:
    for tri, vli, held in splits:
        X_a, y_a, X_b, y_b, _ = preprocess_inside_fold(df.iloc[tri], df.iloc[vli])
        if y_b.nunique() < 2:
            continue
        row = {"held_out_school": held, "n_val": len(y_b),
               "n_val_positive": int(y_b.sum())}
        for arm in HEADLINE:
            _, pr, _ = fit_arm(arm, X_a, y_a, SPLIT_SEED)
            row[arm] = score_binary(y_b, pr(X_b))["auc_pr"]
        row["diff"] = row[HEADLINE[1]] - row[HEADLINE[0]]
        logo.append(row)
    logo = pd.DataFrame(logo)
    logo.to_csv(OUT / "diagnostic_leave_one_school_out.csv", index=False)
    print("\nLEAVE-ONE-SCHOOL-OUT (this is what C2 actually asks for)")
    print(logo.round(4).to_string(index=False))
    print(f"\nsign of the difference consistent across clusters: "
          f"{bool((logo['diff'] > 0).all() or (logo['diff'] < 0).all())}")
    print(f"effective cluster count: {len(logo)}")

full mean -0.00222 (-)
 removed_seed  remaining_mean_diff sign  flipped
           42             -0.00208    -    False
          123             -0.00204    -    False
          456             -0.00232    -    False
          789             -0.00224    -    False
         1024             -0.00203    -    False
         2048             -0.00231    -    False
         3333             -0.00238    -    False
         5555             -0.00214    -    False
         7777             -0.00237    -    False
         9999             -0.00228    -    False
sign flips: 0/10

LEAVE-ONE-SCHOOL-OUT (this is what C2 actually asks for)
held_out_school  n_val  n_val_positive  E_all_ce_noW  G_all_focal_noW    diff
        AYED_RC     79               6        1.0000           1.0000  0.0000
        KNU_JHS    329              11        0.9443           0.9266 -0.0177
            SHI     92              19        0.8733           0.8778  0.0045
           WEWE    481              50        0.964

In [7]:
# ---- 4. Q23's eight conditions, scored from data --------------------
cap = fold_df.groupby("arm")["pred_std"].mean()
ref_auc = fold_df[fold_df["arm"] == HEADLINE[0]]["auc_pr"].mean()
sign_stable = (summ["n_seeds_favouring_ref"] >= 9 or
               summ["n_seeds_favouring_test"] >= 9)

conds = [
 {"id": "(a)", "condition": "Seed count and sign stability",
  "verdict": "PASS" if (len(SEEDS) >= 10 and sign_stable and
                        not summ["sd_exceeds_effect"]) else "FAIL",
  "evidence": f"{len(SEEDS)} seeds, same pipeline as the headline; "
              f"{summ['n_seeds_favouring_ref']}/{summ['n_seeds']} favour the "
              f"reference; between-seed SD {summ['between_seed_sd']:.5f} vs "
              f"effect {abs(summ['grand_mean_diff']):.5f}"},
 {"id": "(b)", "condition": "Tuning parity across arms", "verdict": "PASS",
  "evidence": "zero search trials each, identical config.SHARED_PARAMS; "
              "full search history disclosed in Notebook 6"},
 {"id": "(c)", "condition": "Baseline identity", "verdict": "PASS",
  "evidence": f"{HEADLINE[0]} and {HEADLINE[1]} take the identical feature "
              f"matrix and the identical weighting mechanism and differ only "
              f"in the objective argument. R01 FAILED this: 38 vs 41 columns "
              f"and is_unbalance on one side only"},
 {"id": "(d)", "condition": "Model capacity against n",
  "verdict": "PASS" if cap[HEADLINE[1]] > 0.05 else "FAIL",
  "evidence": f"predicted-probability SD: focal {cap[HEADLINE[1]]:.4f}, "
              f"cross-entropy {cap[HEADLINE[0]]:.4f}, base rate {BASE:.3f}. "
              f"The model expresses variation, so playbook causes 3 and 5 stay "
              f"ruled out on evidence"},
 {"id": "(e)", "condition": "Metric sensitivity to the claimed decision",
  "verdict": "FAIL" if ref_auc > 0.95 else "PASS",
  "evidence": f"reference arm at {ref_auc:.4f} AUC-PR. Above ~0.95 there is "
              f"essentially no headroom for an intervention to move, so a null "
              f"cannot distinguish 'no effect' from 'no room for an effect'"},
 {"id": "(f)", "condition": "Base rate against reported accuracy",
  "verdict": "PASS",
  "evidence": f"base rate {100*BASE:.1f}%, majority-class accuracy "
              f"{100*(1-BASE):.1f}%, achieved recall "
              f"{fold_df[fold_df['arm']==HEADLINE[1]]['recall'].mean():.3f}. "
              f"Not a majority-class artefact"},
 {"id": "(g)", "condition": "Effective n after splitting against raw n",
  "verdict": "FAIL",
  "evidence": f"n={len(df)} raw but only {int(df[TARGET].sum())} positive cases, "
              f"across {df[SCHOOL_COL].nunique() if SCHOOL_COL in df else '?'} "
              f"school clusters, the largest holding "
              f"{100*df[SCHOOL_COL].value_counts(normalize=True).iloc[0]:.0f}% "
              f"of the sample" if SCHOOL_COL in df else "cluster column absent"},
 {"id": "(h)", "condition": "Analysis unit against hypothesis unit",
  "verdict": "PASS",
  "evidence": "the mechanism is per-instance and instance-level evidence is "
              "now reported (Notebook 6b, instance_level_difficulty.csv). "
              "R01 FAILED this: every reported number was fold-averaged"},
]
q23 = pd.DataFrame(conds)
q23.to_csv(OUT / "q23_artefact_or_null.csv", index=False)
print("Q23 — ARTEFACT OR NULL\n")
for c in conds:
    print(f"{c['id']} {c['verdict']:<5s} {c['condition']}")
    print(f"      {c['evidence']}\n")
n_fail = int((q23["verdict"] == "FAIL").sum())
print(f"{n_fail} of 8 conditions fail.")

# Is this a null at all? A stable sign, SD smaller than the effect, and a
# CI excluding zero is a RELIABLE effect, not a null — even if it is small.
IS_NULL = not (sign_stable and not summ["sd_exceeds_effect"]
               and (summ["ci95_hi"] < 0 or summ["ci95_lo"] > 0))
if IS_NULL:
    print("\nVERDICT: the null is " +
          ("UNESTABLISHED, not disproven." if n_fail else "ESTABLISHED."))
else:
    direction = "LOWER" if summ["grand_mean_diff"] < 0 else "HIGHER"
    print(f"""
VERDICT: THIS IS NOT A NULL.
Focal loss gives a small, RELIABLE, {direction} AUC-PR than cross-entropy:
{summ['grand_mean_diff']:+.4f}, 95% CI [{summ['ci95_lo']:+.4f}, {summ['ci95_hi']:+.4f}],
favoured by the reference in {summ['n_seeds_favouring_ref']}/{summ['n_seeds']} seeds.

The failing conditions do not make it a null. They bound how far it can be
read: {', '.join(q23[q23['verdict']=='FAIL']['id'])} — the reference sits near the ceiling, so the
effect is statistically reliable but practically negligible, and it rests
on {int(df[TARGET].sum())} dropout cases in four schools.""")

Q23 — ARTEFACT OR NULL

(a) PASS  Seed count and sign stability
      10 seeds, same pipeline as the headline; 10/10 favour the reference; between-seed SD 0.00121 vs effect 0.00222

(b) PASS  Tuning parity across arms
      zero search trials each, identical config.SHARED_PARAMS; full search history disclosed in Notebook 6

(c) PASS  Baseline identity
      E_all_ce_noW and G_all_focal_noW take the identical feature matrix and the identical weighting mechanism and differ only in the objective argument. R01 FAILED this: 38 vs 41 columns and is_unbalance on one side only

(d) PASS  Model capacity against n
      predicted-probability SD: focal 0.2638, cross-entropy 0.2731, base rate 0.088. The model expresses variation, so playbook causes 3 and 5 stay ruled out on evidence

(e) FAIL  Metric sensitivity to the claimed decision
      reference arm at 0.9869 AUC-PR. Above ~0.95 there is essentially no headroom for an intervention to move, so a null cannot distinguish 'no effect' from 'no ro

In [8]:
# ---- 5. Q24 routing, computed --------------------------------------
pred_std = float(cap[HEADLINE[1]])
route = []
if pred_std < 0.05:
    route.append((1, "Predictions barely vary across cases", "cause 3/5"))
else:
    print(f"row 1 checked and does NOT match: predicted-probability SD "
          f"{pred_std:.4f}. Causes 3 and 5 are ruled out on evidence, which "
          f"narrows the search usefully.")
if ref_auc > 0.95:
    route.append((14, "Metric at ceiling, nothing to move", "cause 14"))
route.append((8, "Mechanism is specific, effect washes out at your level", "cause 12"))
if bool(power_df["underpowered"].iloc[0]):
    route.append((12, "Not significant; cannot separate no-effect from "
                      "not-enough-data", "cause 17 (confirmed, downstream)"))

route = sorted(route)
top = route[0]
if not IS_NULL:
    print("\nNOTE: the Q24 routing table diagnoses NULL results. This result is "
          "a reliable effect, so the routing below is context, not a diagnosis.")
print(f"\nROUTED: \"{top[1]}\"")
print(f"  -> start at {top[2]}")
print(f"  also matching: {[r[2] for r in route[1:]]}")
print(f"  secondary, carried from Q23: causes 7 and 9")
print(f"  ruled out on evidence: causes 3 and 5 (prediction SD {pred_std:.4f})")
pd.DataFrame(route, columns=["row", "symptom", "cause"]).to_csv(
    OUT / "q24_routing.csv", index=False)

row 1 checked and does NOT match: predicted-probability SD 0.2638. Causes 3 and 5 are ruled out on evidence, which narrows the search usefully.

NOTE: the Q24 routing table diagnoses NULL results. This result is a reliable effect, so the routing below is context, not a diagnosis.

ROUTED: "Mechanism is specific, effect washes out at your level"
  -> start at cause 12
  also matching: ['cause 17 (confirmed, downstream)', 'cause 14']
  secondary, carried from Q23: causes 7 and 9
  ruled out on evidence: causes 3 and 5 (prediction SD 0.2638)


In [9]:
# ---- 6. the handoff block ------------------------------------------
handoff = f"""NULL HANDOFF — run {OUT.name}

ROUTED          : "{top[1]}"
                  -> start at {top[2]}, then cause 4 (effective n at the
                     cluster level).
                  Secondary: causes 7, 9. Confirmed downstream: cause 17.
                  Ruled out on evidence: causes 3, 5 (prediction SD {pred_std:.4f}).

CONTRAST        : {HEADLINE[0]} vs {HEADLINE[1]}
                  identical feature matrix, identical weighting, one argument
                  different. R01's contrast varied three things.

TEN SEEDS       : grand mean {summ['grand_mean_diff']:+.5f}
                  between-seed SD {summ['between_seed_sd']:.5f}
                  95% CI [{summ['ci95_lo']:+.5f}, {summ['ci95_hi']:+.5f}]
                  favouring focal {summ['n_seeds_favouring_test']}/{summ['n_seeds']}
                  seeds with p < {ALPHA_LEVEL}: {summ['n_seeds_p_below_alpha']}

POWER           : MDE {MDE:.4f} vs observed {observed:.4f} -> \
{'UNDERPOWERED' if observed < MDE else 'adequate'}

Q23             : {n_fail} of 8 conditions fail
RESULT TYPE     : {'NULL (unestablished)' if IS_NULL and n_fail else 'NULL' if IS_NULL else 'RELIABLE EFFECT, near ceiling'}

PIPELINE        : all preprocessing in-fold; CV on the training pool only;
                  test partition scored once in Notebook 8 after freeze.

PERMANENT       : the R01 test partition was scored across six notebooks.
                  No re-run repairs the history of a partition. State it in
                  the limitations.
"""
print(handoff)
(OUT / "NULL_HANDOFF.txt").write_text(handoff)

write_manifest(OUT, {
    "notebook": "09_diagnostic", "test_set_scored": False,
    "seeds": SEEDS, "arms": list(HEADLINE),
    "grand_mean_diff": float(summ["grand_mean_diff"]),
    "between_seed_sd": float(summ["between_seed_sd"]),
    "mde": float(MDE), "observed": float(observed),
    "underpowered": bool(observed < MDE),
    "q23_conditions_failed": n_fail,
    "is_null": bool(IS_NULL),
    "null_unestablished": bool(IS_NULL and n_fail > 0),
    "seed_loo_flips": int(loo["flipped"].sum()),
    "committed_diagnostics": ["diagnostic_10_seed_stability.csv",
                              "diagnostic_power_analysis.csv",
                              "diagnostic_leave_one_seed_out.csv",
                              "diagnostic_leave_one_school_out.csv"],
})
print("All four diagnostic CSVs are written to the run directory. COMMIT THEM "
      "— Q4 failed because Table 5, the MDE and the zero-flip claim traced to "
      "nothing in the repository.")

NULL HANDOFF — run 20260922T231411Z_records_plus_questionnaire

ROUTED          : "Mechanism is specific, effect washes out at your level"
                  -> start at cause 12, then cause 4 (effective n at the
                     cluster level).
                  Secondary: causes 7, 9. Confirmed downstream: cause 17.
                  Ruled out on evidence: causes 3, 5 (prediction SD 0.2638).

CONTRAST        : E_all_ce_noW vs G_all_focal_noW
                  identical feature matrix, identical weighting, one argument
                  different. R01's contrast varied three things.

TEN SEEDS       : grand mean -0.00222
                  between-seed SD 0.00121
                  95% CI [-0.00312, -0.00136]
                  favouring focal 0/10
                  seeds with p < 0.05: 4

POWER           : MDE 0.0039 vs observed 0.0022 -> UNDERPOWERED

Q23             : 2 of 8 conditions fail
RESULT TYPE     : RELIABLE EFFECT, near ceiling

PIPELINE        : all preprocessing in-fold

---

## Before you submit

Drive has already saved everything — nothing to push. But two things still
have to happen before submission, and neither is automatic.


In [10]:
# ---- what this run produced, and what is still owed ----
import os, sys
from pathlib import Path

try:
    latest = sorted(Path(OUT).parent.glob("*"))[-1]
    files = sorted(p.relative_to(OUT).as_posix() for p in Path(OUT).rglob("*")
                   if p.is_file())
    print(f"run directory : {Path(OUT).relative_to(REPO)}")
    print(f"files written : {len(files)}")
    for f in files:
        print("   ", f)
except Exception as e:
    print("no run directory recorded in this session:", e)

print("""
────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from a clone unchanged.

   Do NOT upload:  data-raw/, data-processed/cleaned_data.csv,
                   any *_snapshot.csv
   DO upload:      config.py, losses.py, pipeline.py, notebooks/,
                   requirements.txt, README.md, and all of results/

2. SET config.FREEZE_TAG BEFORE SCORING THE TEST SET.
   Without git there is no commit hash to anchor the freeze to. Put a
   fixed dated string in config.py — e.g. "R02-freeze-2026-09-25-1430" —
   at the moment you freeze the configuration, and never revise it.
   Notebook 8 refuses to score the test set until it is set.
────────────────────────────────────────────────────────────────────""")


run directory : results/notebook09_diagnostic/20260922T231411Z_records_plus_questionnaire
files written : 11
    NULL_HANDOFF.txt
    RUN_MANIFEST.json
    diagnostic_10_seed_stability.csv
    diagnostic_fold_scores.csv
    diagnostic_leave_one_school_out.csv
    diagnostic_leave_one_seed_out.csv
    diagnostic_power_analysis.csv
    environment_versions.csv
    pip_freeze.txt
    q23_artefact_or_null.csv
    q24_routing.csv

────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from a clone unchanged.

   Do NOT upload:  data-raw/, data-